# From Logistic Regression to a Neural Network - Concepts

Can we identify spam without flagging too many legitimate emails? One observation is one email: **0 = ham, 1 = spam**. A prediction supports review, not automatic deletion.

In this notebook, we will:
1. Explain a sigmoid output neuron and its predicted probability.
2. Explain what a hidden ReLU layer adds to the same input features.
3. Compare predictions with a simple baseline without data leakage.
4. Connect ranking metrics, decision thresholds and business costs.

This accompanies **SW07 Lecture 1: From Linear Models to Neural Networks**. Prerequisites are logistic regression, training/validation/test sets and basic Python. Read the idea, formula and small example in that order; demonstration code is provided. One example fits the lecture's 10-minute demonstration slot. The complete guided practice belongs in a separate practical session.

Continue with the [exercise](logistic_regression_to_neural_network_exercise.ipynb), then check the [solution](logistic_regression_to_neural_network_solution.ipynb).

## Libraries and settings

The small examples use supplied weights and hypothetical errors, not fitted spam models. The exercise uses the repository's prepared email CSV. No dataset download or raw email text is needed for the core practical.

In [1]:
import numpy as np
import pandas as pd

SEED = 7
np.random.seed(SEED)

## 1. Logistic regression is a sigmoid output neuron

Multiply each scaled input by its weight, add a bias, then apply the sigmoid:

$$z=\mathbf{w}^{T}\mathbf{x}+b,\qquad p=\frac{1}{1+\exp(-z)}.$$

Here $\mathbf{x}$ contains inputs, $\mathbf{w}$ contains their learned weights, $b$ is the bias and $p$ is the estimated spam probability. For the practical's 16 features, there are **17 parameters** including the bias. A probability is not yet a review decision.

For two toy inputs $[2,-1]$, weights $[0.5,1]$ and bias 0, the score is $2(0.5)-1(1)=0$ and the probability is 0.5. The following code illustrates prediction, not fitting.

In [2]:
inputs = np.array([2.0, -1.0])
weights = np.array([0.5, 1.0])
bias = 0.0
score = inputs @ weights + bias
probability = np.exp(-np.logaddexp(0, -score))
print('Score:', score, '| Probability:', probability)

Score: 0.0 | Probability: 0.5


## 2. Add a hidden layer, not new information

A hidden layer learns combinations of the same numeric inputs:

$$\mathbf{h}=\operatorname{ReLU}(\mathbf{x}W_h+\mathbf{b}_h),\qquad p=\operatorname{sigmoid}(\mathbf{h}\mathbf{w}_o+b_o).$$

$W_h$ and $\mathbf{b}_h$ are hidden weights and biases; $\mathbf{w}_o$ and $b_o$ are output weights and bias. **ReLU** replaces a negative score with zero and keeps a positive score unchanged. Multiple units create a nonlinear prediction function. Without a nonlinear activation, stacked linear layers collapse to a single linear transformation.

With **16 inputs, 8 hidden units and one output**, there are $16\times8+8+8\times1+1=145$ parameters, including biases. More units do not guarantee better generalization. The inputs remain counts and ratios: the network does not suddenly understand email text.

The toy example below uses one hidden unit: hidden score $2(0.5)-1(1)+0.5=0.5$; output score $0.5(2)-0.5=0.5$; probability approximately 0.6225. Its weights are supplied, not trained. Backpropagation and gradient updates are covered in SW07 Lecture 2.

In [3]:
hidden_weights = np.array([[0.5], [1.0]])
hidden_bias = np.array([0.5])
hidden = np.maximum(0, inputs @ hidden_weights + hidden_bias)
output_score = hidden @ np.array([2.0]) - 0.5
network_probability = np.exp(-np.logaddexp(0, -output_score))
print('Hidden activation:', hidden, '| Probability:', network_probability)
print('Parameters for 16 -> 8 -> 1:', 16 * 8 + 8 + 8 * 1 + 1)

Hidden activation: [0.5] | Probability: 0.6224593312018546
Parameters for 16 -> 8 -> 1: 145


## 3. Establish a baseline and keep evaluation separate

The prior baseline assigns every email the **training-set spam proportion**. At threshold 0.5 it can flag no emails when spam is the minority. This may look cheap but gives zero spam recall.

The practical reads [email_spam.csv](../../../00_Data/email_spam.csv): **18,000 training, 6,000 validation and 6,000 test rows** in stored partitions. Both models use the same 16 deterministic count and ratio features. `label`, `split`, `source_row` and `message_key` must not become predictors. The features do not include raw subjects or bodies.

Fit preprocessing and models only on training rows. Reuse the fitted scaler for validation and test rows; do not fit a separate scaler on each partition. Choose hidden size and decision thresholds on validation data. Freeze choices before test evaluation. These historical test results have been inspected in earlier work, so re-running them is not fresh independent confirmation.

## 4. Ranking quality is not business value

- **Precision:** fraction of flagged messages that really are spam.
- **Recall:** fraction of all spam that is flagged.
- **Average precision (AP):** a summary of precision and recall across score thresholds; higher values indicate better ranking on the evaluated data. It does not establish probability calibration or cost savings.
- **Threshold:** the probability cutoff that turns a score into a review decision.

For illustration, a false positive (FP: legitimate email flagged) costs 5 units and a false negative (FN: spam missed) costs 1:

$$\text{assumed cost}=5\,\mathrm{FP}+\mathrm{FN}.$$

Changing the threshold changes FP and FN, not the ranking itself. The next table is **hypothetical**, not measured model performance. Policy A has more false positives but lower total assumed cost; neither error type alone determines the best policy.

In [4]:
policies = pd.DataFrame({'Policy': ['A', 'B'],
                         'False positives': [2, 0],
                         'False negatives': [10, 25]}).set_index('Policy')
policies['Assumed total cost'] = 5 * policies['False positives'] + policies['False negatives']
print(policies.to_string())

        False positives  False negatives  Assumed total cost
Policy                                                      
A                     2               10                  20
B                     0               25                  25


## 5. Summary and limits

Logistic regression uses a sigmoid output neuron; an MLP adds nonlinear hidden units while keeping the same 16 inputs. Start with a baseline, share training-fitted preprocessing, choose settings on validation data, then evaluate fixed choices. Read AP, precision, recall and assumed costs together.

- Standardized logistic coefficients describe conditional model associations, not causes. Correlated inputs make attribution ambiguous.
- Counts and ratios miss message meaning. More parameters cannot restore information absent from the inputs.
- Training loss is not evidence of generalization. Check convergence warnings and validation results; neither model is guaranteed to win.
- Historical random splits can mix related campaigns. They do not establish future-email performance. Probability calibration, review capacity and monitoring need separate checks.
- Features and hashed identifiers are not a guarantee of anonymity. Protect data and investigate errors before operational use.

### Optional practical material

Optional extensions include XGBoost, model persistence, pretrained-text features and an additional hidden layer. These are not required for the core exercise. XGBoost and text embeddings need additional packages; embeddings also require authorized private raw emails and verified reference splits. Model export and synthetic-message probes are operational illustrations, not independent evaluation.

## References

**Required student reading:** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning: With Applications in Python*. Springer. Selected classification and neural-network material. https://www.statlearning.com/

**Optional reading / instructor reference:** Molnar, C. *Interpretable Machine Learning*, 2nd ed. Coefficients and interpretation limitations. https://christophm.github.io/interpretable-ml-book/

**Official technical documentation:** [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html), [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).

**Dataset:** NIKHIL CHAUDHARY 7108. (n.d.). *Spam email dataset (Enron, TREC and others)*. [Kaggle](https://www.kaggle.com/datasets/nikhilchaudhary7108/spam-email-datasetenron-trec-and-others). Combined-source upstream rights and labels have not been independently audited. The practical uses the local feature-only CSV, not an upstream download. See [the repository README](../../../README.md) for preparation and limitations, and [requirements.txt](../../../requirements.txt) for the environment.

### Jupyter notebook - footer info

Include this information when submitting your notebook.

In [5]:
import platform
from datetime import datetime

print(platform.platform())
print('Python:', platform.python_version())
print('Date:', datetime.now().isoformat(timespec='seconds'))

macOS-26.6.2-x86_64-i386-64bit
Python: 3.9.13
Date: 2026-10-25T19:06:37
